# Panel mixed logit

## Model

Each individual has one draw-specific taste vector shared across all repeated
choice occasions $t\in\mathcal{P}_n$:

$$
\boldsymbol\beta_{nr}
=\boldsymbol\mu+\boldsymbol L\boldsymbol z_r,
$$

$$
L_n=
\int\prod_{t\in\mathcal{P}_n}
P_{nt,y_{nt}}(\boldsymbol\beta_n)
f(\boldsymbol\beta_n)\,d\boldsymbol\beta_n
\approx
\frac1R\sum_{r=1}^{R}
\prod_{t\in\mathcal{P}_n}P_{nt,y_{nt},r}.
$$

The example generates three choice occasions per individual and estimates
correlated random time and cost coefficients using panel likelihood
integration.

Machine configuration: AMD Ryzen 9 9950X3D CPU (16 cores), 64 GB RAM, and
NVIDIA GeForce RTX 5090 GPU (32 GB VRAM), running Ubuntu 24.04.4 with
PyTorch 2.12.1 and CUDA 13.0. Install the released package with
`pip install torchdcm`, then select CPU or CUDA through the `device` argument.


In [1]:
import numpy as np
import torch

from IPython.display import HTML, display

from torchdcm import Beta, ChoiceDataset, MixedLogit, RandomCoefficient, UtilitySpec
from torchdcm.datasets import make_swissmetro_like
# Use double precision and a fixed seed for stable, reproducible estimation.
torch.set_default_dtype(torch.float64)
torch.manual_seed(7)
# Use CUDA automatically when available; set device = "cpu" to force CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"TorchDCM example running on {device}")


TorchDCM example running on cuda


In [2]:
rng = np.random.default_rng(23)
# Generate eight repeated choice occasions per individual.
frame = make_swissmetro_like(n_obs=960, seed=23)
frame["person_id"] = np.arange(len(frame)) // 8
alternatives = np.array(["TRAIN", "SM", "CAR"])
time = frame[["time_train", "time_sm", "time_car"]].to_numpy()
cost = frame[["cost_train", "cost_sm", "cost_car"]].to_numpy()
available = frame[["avail_train", "avail_sm", "avail_car"]].to_numpy(bool)

# Map rows to individuals so tastes remain shared within each panel.
person_codes, person_inverse = np.unique(
    frame["person_id"].to_numpy(),
    return_inverse=True,
)
sd_time, sd_cost, correlation = 0.015, 0.040, 0.40
# Construct the correlated time-cost taste covariance matrix.
covariance = np.array(
    [
        [sd_time**2, correlation * sd_time * sd_cost],
        [correlation * sd_time * sd_cost, sd_cost**2],
    ]
)
# Draw one random coefficient vector for every individual.
person_tastes = rng.multivariate_normal(
    mean=[-0.025, -0.070],
    cov=covariance,
    size=len(person_codes),
)
random_tastes = person_tastes[person_inverse]
utility = (
    np.array([0.25, 0.0, 0.15])
    + random_tastes[:, [0]] * time
    + random_tastes[:, [1]] * cost
)
masked = np.where(available, utility, -np.inf)
probabilities = np.exp(masked - np.max(masked, axis=1, keepdims=True))
probabilities /= probabilities.sum(axis=1, keepdims=True)
frame["choice"] = [
    rng.choice(alternatives, p=row) for row in probabilities
]

# Declare person_id so TorchDCM detects and compiles the panel.
data = ChoiceDataset.from_wide(
    frame,
    alternatives=alternatives.tolist(),
    choice="choice",
    variables={
        "time": {"TRAIN": "time_train", "SM": "time_sm", "CAR": "time_car"},
        "cost": {"TRAIN": "cost_train", "SM": "cost_sm", "CAR": "cost_car"},
    },
    availability={
        "TRAIN": "avail_train",
        "SM": "avail_sm",
        "CAR": "avail_car",
    },
    individual_id="person_id",
)
# Specify mean utilities before attaching random coefficients.
spec = UtilitySpec()
spec.utility(
    "TRAIN",
    Beta("ASC_TRAIN")
    + Beta("B_TIME", init=-0.02) * "time"
    + Beta("B_COST", init=-0.05) * "cost",
)
spec.utility(
    "SM",
    Beta("B_TIME", init=-0.02) * "time"
    + Beta("B_COST", init=-0.05) * "cost",
)
spec.utility(
    "CAR",
    Beta("ASC_CAR")
    + Beta("B_TIME", init=-0.02) * "time"
    + Beta("B_COST", init=-0.05) * "cost",
)
print(
    f"Choice occasions: {data.n_obs}; "
    f"individuals: {data.n_individuals}; "
    f"panel detected: {data.has_panel}"
)


Choice occasions: 960; individuals: 120; panel detected: True


In [3]:
# Multiply probabilities within person before averaging over draws.
model = MixedLogit(
    spec,
    [
        RandomCoefficient("B_TIME", sigma_init=0.010),
        RandomCoefficient("B_COST", sigma_init=0.030),
    ],
    n_draws=48,
    seed=23,
    antithetic=True,
    panel=True,
    correlated=True,
    device=device,
    max_iter=100,
)
result = model.fit(data)
# Estimate the panel likelihood and render the structured report.
display(HTML(result.report().to_html()))


Model family,MixedLogit
Estimation objective,Maximum simulated log likelihood
TorchDCM version,0.1.0
PyTorch version,2.12.1+cu130
Python version,3.12.13
Operating system,Linux-6.17.0-35-generic-x86_64-with-glibc2.39
Device,cuda
Tensor dtype,float64
Optimizer,torch.optim.LBFGS
Maximum iterations,100
Line search,strong_wolfe
